# VIVO Backgammon — Entrenamiento red ancha en GPU (Colab)

**Antes de ejecutar:**
1. La rama `260815-wide-net-gpu` ya está subida a GitHub (hecho por Hermes).
2. `Runtime` → `Change runtime type` → `GPU` (T4/A100).
3. Ejecuta las celdas en orden. La celda 5 se detiene SOLA al llegar a 60% (auto-stop en cli.ts).
4. Descarga `model_weights.json` (celda 6) y colócalo en `public/` del proyecto local.

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Instalar Node 20 (LTS, compatible con tfjs-node-gpu)


In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs
!node --version && npm --version


## 3. Clonar la rama de trabajo

REPO_URL ya apunta al repo correcto (Pirzl/Backgammon-AR-Pro, rama 260815-wide-net-gpu).

In [ ]:
REPO_URL = "https://github.com/Pirzl/Backgammon-AR-Pro.git"
BRANCH = "260815-wide-net-gpu"

!rm -rf vivo_train_repo
!git clone --branch {BRANCH} {REPO_URL} vivo_train_repo
%cd vivo_train_repo

## 4. Instalar dependencias (tfjs-node-gpu enlaza con CUDA de Colab)

In [ ]:
!npm ci
!npm rebuild @tensorflow/tfjs-node-gpu
!node -e "const tf=require('@tensorflow/tfjs-node-gpu'); console.log('BACKEND=', tf.getBackend());"


In [ ]:
# Verificacion GPU: si imprime BACKEND= tensorflow -> GPU activa.
# Si imprime error o BACKEND= cpu/webgl -> la GPU NO esta disponible, cancela.
!node -e "const tf=require('@tensorflow/tfjs-node-gpu'); const b=tf.getBackend(); console.log('BACKEND=', b); if(b!=='tensorflow'){console.error('GPU NO ACTIVA'); process.exit(1);}"


## 5. Entrenar (self-play on-policy, vs heurística)

**ANTES de ejecutar:** la celda anterior debe imprimir . Si no, la GPU no está activa (cancelar, reconectar GPU en Runtime > Change runtime type).

Corre hasta que **dos  consecutivos** den  y entonces se detiene SOLO (auto-stop en cli.ts), guardando .
No hace falta pulsar Interrupt: cuando llegue al 60% el proceso termina.

Si plateau < 0.60 tras ~5000 partidas: detén, cambia  a  en , push, reinicia desde celda 3.


In [ ]:
!npx tsx src/features/ai-worker/training/cli.ts \
  --games=100000 --opponent=heuristic --label=outcome \
  --exploration=0.15 --max-moves=400 --epochs=3 \
  --eval-every=250 --eval-games=200 --nn-blend=1 --save-every=50 --stop-rate=0.60 --stop-streak=2

## 6. Descargar pesos entrenados

Ejecuta tras que la celda 5 termine sola (los pesos se guardan con --save-every=50).

In [ ]:
from google.colab import files
files.download('public/model_weights.json')

## 7. Verificar forma del peso (opcional, local)

En tu PC, tras colocar el archivo en `public/model_weights.json`:
```bash
npx tsx src/features/ai-worker/training/tournament.ts   # debe dar rate >= 0.60 (PASS)
npx vite build                                          # build OK
```
La capa 0 debe tener `shape:[198,256]` (no `[198,40]`).